# CDC PLACES Cleaning

## Purpose
This notebook cleans and prepares the CDC PLACES dataset for later integration with BRFSS 2023 data. The focus is on selecting relevant health indicators, fixing data types, checking missing values, and creating a state-level summary file for later analysis.

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
#Load data

PLACES = pd.read_csv(r"C:\Users\alin2\OneDrive\Documents\Behavioral Risk & Health Outcomes Analysis (BRFSS 2023)\data\PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260228.csv")
PLACES.head()

C:\Users\alin2\AppData\Local\Temp\ipykernel_19700\869668532.py:3: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  PLACES = pd.read_csv(r"C:\Users\alin2\OneDrive\Documents\Behavioral Risk & Health Outcomes Analysis (BRFSS 2023)\data\PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260228.csv")


,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
0,2023,AR,Arkansas,Drew,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,29.9,...,26.6,33.3,"16,945","13,230",5043,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-91.7196579038053 33.5894113647675)
1,2023,AR,Arkansas,Fulton,BRFSS,Health Outcomes,Current asthma among adults,%,Crude prevalence,10.6,...,9.2,11.9,"12,421","9,795",5049,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,POINT (-91.817888079321 36.3816206347765)
2,2023,AR,Arkansas,Howard,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,31.2,...,27.7,34.8,"12,533","9,311",5061,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-93.9938053308515 34.0889044688856)
3,2023,AR,Arkansas,Miller,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,4.7,...,4.2,5.3,"42,415","32,396",5091,HLTHOUT,STROKE,CrdPrv,Stroke,POINT (-93.8924281761086 33.3123156126392)
4,2023,AR,Arkansas,Ouachita,BRFSS,Disability,Any disability among adults,%,Crude prevalence,42.8,...,37.9,47.8,"21,793","16,948",5103,DISABLT,DISABILITY,CrdPrv,Any Disability,POINT (-92.882040791321 33.593253927504)


In [7]:
#Check shape
PLACES.shape

(229298, 22)

In [8]:
#Check columns
PLACES.columns.tolist()

['Year',
 'StateAbbr',
 'StateDesc',
 'LocationName',
 'DataSource',
 'Category',
 'Measure',
 'Data_Value_Unit',
 'Data_Value_Type',
 'Data_Value',
 'Data_Value_Footnote_Symbol',
 'Data_Value_Footnote',
 'Low_Confidence_Limit',
 'High_Confidence_Limit',
 'TotalPopulation',
 'TotalPop18plus',
 'LocationID',
 'CategoryID',
 'MeasureId',
 'DataValueTypeID',
 'Short_Question_Text',
 'Geolocation']

## Initial Dataset Overview

The CDC PLACES dataset contains county-level public health estimates for the United States. Each record represents a specific health indicator measured within a county, along with its corresponding value and confidence intervals.

The dataset includes geographic identifiers (state and county), indicator descriptions (Measure), and numerical estimates (Data_Value), which represent the prevalence of specific health conditions or behaviors.

For this project, the dataset will be cleaned and reduced to a focused set of health indicators that align with the research goal. Since the analysis aims to connect individual behavioral patterns (from BRFSS) to broader population-level health outcomes, this dataset will later be aggregated from county-level to state-level.

This step ensures that the data is structured, relevant, and ready for integration with BRFSS in subsequent analysis.

In [ ]:
# Check available measures
PLACES['Measure'].unique()

array(['Arthritis among adults', 'Current asthma among adults',
       'Stroke among adults', 'Any disability among adults',
       'Obesity among adults', 'Diagnosed diabetes among adults',
       'Depression among adults', 'Hearing disability among adults',
       'Binge drinking among adults', 'High blood pressure among adults',
       'Vision disability among adults',
       'Mobility disability among adults',
       'Self-care disability among adults',
       'Frequent mental distress among adults',
       'High cholesterol among adults who have ever been screened',
       'Frequent physical distress among adults',
       'Colorectal cancer screening among adults aged 45–75 years',
       'No leisure-time physical activity among adults',
       'Chronic obstructive pulmonary disease among adults',
       'Independent living disability among adults',
       'Coronary heart disease among adults',
       'Mammography use among women aged 50-74 years',
       'Visits to doctor for rou

## Selecting Relevant Health Indicators

The PLACES dataset contains many public health measures, but not all of them are needed for this project. To keep the analysis focused, this notebook selects a smaller group of indicators that align with the project question.

The selected measures include both behavioral risk indicators and broader health outcomes:

- Obesity among adults  
- Diagnosed diabetes among adults  
- No leisure-time physical activity among adults  
- Current cigarette smoking among adults  
- Binge drinking among adults  

These indicators will later be compared with BRFSS response patterns after aggregation to the state level.

In [11]:
PLACES_clean = PLACES[[
    'Year',
    'StateAbbr',
    'StateDesc',
    'LocationName',
    'Measure',
    'Data_Value'
]].copy()

PLACES_clean.head()

,Year,StateAbbr,StateDesc,LocationName,Measure,Data_Value
0,2023,AR,Arkansas,Drew,Arthritis among adults,29.9
1,2023,AR,Arkansas,Fulton,Current asthma among adults,10.6
2,2023,AR,Arkansas,Howard,Arthritis among adults,31.2
3,2023,AR,Arkansas,Miller,Stroke among adults,4.7
4,2023,AR,Arkansas,Ouachita,Any disability among adults,42.8


In [12]:
#Check data types
PLACES_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229298 entries, 0 to 229297
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Year          229298 non-null  int64  
 1   StateAbbr     229298 non-null  object 
 2   StateDesc     229298 non-null  object 
 3   LocationName  229218 non-null  object 
 4   Measure       229298 non-null  object 
 5   Data_Value    229232 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 10.5+ MB


In [14]:
#Convert Data_Value to numeric
PLACES_clean['Data_Value'] = pd.to_numeric(PLACES_clean['Data_Value'], errors='coerce')

PLACES_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229298 entries, 0 to 229297
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Year          229298 non-null  int64  
 1   StateAbbr     229298 non-null  object 
 2   StateDesc     229298 non-null  object 
 3   LocationName  229218 non-null  object 
 4   Measure       229298 non-null  object 
 5   Data_Value    229232 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 10.5+ MB


### Why the Data_Value column was converted

The `Data_Value` column contains the main health estimate used in this project. Even though it was already read as numeric in this file, the conversion step was still included as a data-cleaning check to make sure the column is safe for later calculations such as grouping, averaging, and visualization.